# 06 · BM25 search

The search bar on the page ranks articles with BM25. The code is my BM25 implementation from Info 202 HW4 (`tokenize`, `Index`, `idf`, `term_score`, `bm25`, `snippet`, `search`) with two changes: it reads the project's articles table instead of `articles.xlsx`, and each document is the article's title plus its first paragraph instead of just the paragraph. Titles matter for a search bar: 70 of the 789 titles have words that don't appear in their own first paragraph (with paragraphs only, searching "Agincourt" ranked *Battle of Agincourt* 4th). All 789 articles are indexed, seed included, so the scores are slightly different from HW4.

The page can't run Python, so the last cell exports the index to `docs/search.js` and the page scores queries with the same formula in JavaScript.

In [1]:
import json
import math
import re
import time
from array import array
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
df = pd.read_parquet(ROOT / "data" / "processed" / "articles.parquet")
df["first_paragraph"] = df["first_paragraph"].fillna("")
print(df.shape)

(789, 13)


## From HW4: tokenizer and index

In [2]:
data = df['title'] + '. ' + df['first_paragraph']   # HW4: df['first_paragraph']
titles = df['title']
TOKEN_RE = re.compile(r"[a-z0-9]+(?:'[a-z]+)?")
def tokenize(text):
    return TOKEN_RE.findall(text.lower())

In [3]:
class Index:
    def __init__(self, texts):
        self.texts = texts
        self.N = len(texts)
        self.length = np.zeros(self.N)              # |D|: number of tokens in each document
        doc_ids, counts = defaultdict(lambda: array("I")), defaultdict(lambda: array("I"))
        for i, text in enumerate(texts):
            tf = Counter(tokenize(text))
            self.length[i] = sum(tf.values())
            for word, count in tf.items():
                doc_ids[word].append(i)
                counts[word].append(count)
        # postings[word] = (array of doc ids, array of term frequencies)
        self.postings = {w: (np.array(doc_ids[w], dtype=np.intp),
                             np.array(counts[w], dtype=float)) for w in doc_ids}
        self.avgdl = self.length.mean()

    def df(self, word):
        # document frequency: how many documents contain the word
        return len(self.postings[word][0]) if word in self.postings else 0

t0 = time.time()
idx = Index(data)
print(f"Indexed in {time.time() - t0:.1f}s.  {len(idx.postings):,} distinct words.")
print(f"Average document length: {idx.avgdl:.3f} words.  "
      f"Shortest: {idx.length.min():.0f}.  Longest: {idx.length.max():.0f}.  Median: {np.median(idx.length):.0f}.")

Indexed in 0.0s.  8,161 distinct words.
Average document length: 79.862 words.  Shortest: 12.  Longest: 275.  Median: 73.


## From HW4: BM25 scoring and search

In [4]:
def idf(idx, word):
    return math.log(idx.N / idx.df(word))

def term_score(tf, doc_len, idf_value, avgdl, k1=1.2, b=0.75):
    length_norm = k1 * (1 - b + b * doc_len / avgdl)
    return idf_value * (tf * (k1 + 1)) / (tf + length_norm)

def bm25(idx, query, k1=1.2, b=0.75):
    scores = np.zeros(idx.N)
    for word in set(tokenize(query)):
        if word not in idx.postings:
            continue
        doc_ids, tf = idx.postings[word]
        scores[doc_ids] += term_score(tf, idx.length[doc_ids], idf(idx, word), idx.avgdl, k1, b)
    return scores

def snippet(text, query, width=170):
    pats = [r"\b" + re.escape(w) + r"\b" for w in set(tokenize(query))]
    m = re.search("|".join(pats), text, re.I)
    if not m:
        return text[:width]
    start = max(0, m.start() - width // 3)
    window = re.sub("|".join(pats), lambda x: f"**{x.group(0)}**", text[start:start + width], flags=re.I)
    return ("..." if start else "") + window + ("..." if start + width < len(text) else "")

def search(query, k=10, k1=1.2, b=0.75):
    if not tokenize(query):
        print(f"No words found in {query!r}. Replace it with a real query.")
        return []
    scores = bm25(idx, query, k1, b)
    n_match = int((scores > 0).sum())
    print(f"query: {query!r}   k1={k1}  b={b}   {n_match:,} documents contain at least one query word")
    for word in sorted(set(tokenize(query))):
        note = f"idf={idf(idx, word):.2f}, in {idx.df(word):,} docs" if idx.df(word) else "NOT IN COLLECTION"
        print(f"   {word:<12} {note}")
    print()
    results = []
    for rank, d in enumerate(np.argsort(-scores)[:k], 1):
        if scores[d] == 0:
            break
        print(f"{rank:>2}. {titles[d]}   [doc {d}]   score {scores[d]:.2f}, {idx.length[d]:.0f} words")
        print(f"      {snippet(idx.texts[d], query)}")
        results.append((int(d), titles[d], float(scores[d])))
    return results

In [5]:
results = search("mad king", k=5)
search("Agincourt", k=3);

query: 'mad king'   k1=1.2  b=0.75   155 documents contain at least one query word
   king         idf=1.63, in 155 docs
   mad          idf=6.67, in 1 docs

 1. Charles VI of France   [doc 150]   score 9.13, 62 words
      ...ved (French: le Bien-Aimé) and in the 19th century, the **Mad** (French: le Fol or le Fou), was **King** of France from 1380 until his death in 1422. He is known for his mental i...
 2. Henry II of Castile   [doc 319]   score 2.81, 70 words
      ...ámara or the Fratricidal (el Fratricida), was the first **King** of Castile and León from the House of Trastámara. He became **king** in 1369 by defeating his half-brother **King** P...
 3. John I of France   [doc 418]   score 2.70, 118 words
      ...ean I le Posthume, Occitan: Joan I lo Postume), was the **King** of France and Navarre, as the posthumous son and successor of Louis X, for the four days he lived in 1316. He...
 4. Charles IV of France   [doc 146]   score 2.60, 74 words
      ...France and the Bald (el 

## Export for the page

`docs/search.js` has what the page needs to score a query: N, avgdl, each document's title and length, and the postings (word → document ids and term frequencies). The page matches results to its articles by title.

In [6]:
export = {
    "N": idx.N,
    "avgdl": idx.avgdl,
    "titles": titles.tolist(),
    "length": idx.length.astype(int).tolist(),
    "postings": {w: [ids.tolist(), tf.astype(int).tolist()] for w, (ids, tf) in idx.postings.items()},
}
out = ROOT / "docs" / "search.js"
out.write_text("window.SEARCH = " + json.dumps(export, ensure_ascii=False, separators=(",", ":")) + ";\n", encoding="utf-8")
print(f"wrote {out.relative_to(ROOT)}: {len(export['postings']):,} words, {out.stat().st_size / 1e6:.2f} MB")

wrote docs/search.js: 8,161 words, 0.39 MB
